# Task 2b: Specialist Restoration Autoencoders (Simple Baseline)

This notebook trains the **three specialist restoration autoencoders** independently:
1. **Salt-and-Pepper Specialist** ($D_{salt}$)
2. **Gaussian Blur Specialist** ($D_{blur}$)
3. **Occlusion Specialist** ($D_{occlusion}$)

### Mathematical Formulation
Each specialist $k \in \{\text{salt}, \text{blur}, \text{occlusion}\}$ is trained strictly on its designated corruption domain to reconstruct the clean target image $x$:
$$\hat{x}_k = D_k\big(E_k(\tilde{x}_k)\big)$$
Each specialist optimizes the composite structural reconstruction objective:
$$\mathcal{L}_{specialist} = \alpha\,\mathcal{L}_{L1}(x,\hat{x}_k) + (1-\alpha)\big(1-\mathrm{SSIM}(x,\hat{x}_k)\big)$$
with initial $\alpha = 0.8$.

In [ ]:
# 1. Config
BATCH_SIZE = 32
LR = 1e-3
LATENT_DIM = 128
BASE_CHANNELS = 32
ALPHA = 0.8
EPOCHS = 15
SEED = 42


In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_dae import ConvDAE
from src.train_loop import train_one_epoch_dae, validate_dae, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device(verbose=True)
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

In [ ]:
# 3. Specialist Training Function
def train_specialist(corruption_type: str, epochs: int = EPOCHS):
    print(f"\nTraining specialist: {corruption_type}")
    train_ds = PetDataset(mode="train", corruption_mode=corruption_type, seed=SEED)
    val_ds = PetDataset(mode="val", corruption_mode=corruption_type, seed=SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model = ConvDAE(base_channels=BASE_CHANNELS, latent_dim=LATENT_DIM, dropout=0.1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    best_loss = float("inf")
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch_dae(model, train_loader, optimizer, device, alpha=ALPHA, scaler=scaler)
        vl = validate_dae(model, val_loader, device, alpha=ALPHA)
        is_best = vl["val_loss"] < best_loss
        best_loss = min(best_loss, vl["val_loss"])
        ckpt = create_checkpoint_dict(
            model, optimizer, epoch, vl["val_loss"], random_seed=SEED,
            extra_metadata={"alpha": ALPHA, "latent_dim": LATENT_DIM,
                            "base_channels": BASE_CHANNELS},
        )
        save_checkpoint(
            checkpoint_dir=checkpoint_dir, prefix=f"task2b_specialist_{corruption_type}",
            ckpt_dict=ckpt, is_best=is_best, task_name=f"task2b_{corruption_type}",
            phase_name="simple", device_name=device_report()["device_name"],
        )
        print(f"[{corruption_type} {epoch:02d}] Train {tr['train_loss']:.4f}; "
              f"Val {vl['val_loss']:.4f}; SSIM {vl['val_ssim']:.4f}")
    return model


In [ ]:
# 4. Train All Three Specialists
salt_model = train_specialist("salt", epochs=EPOCHS)
blur_model = train_specialist("blur", epochs=EPOCHS)
occl_model = train_specialist("occlusion", epochs=EPOCHS)
print("\nAll 3 specialists trained and checkpoints saved successfully!")

In [ ]:
# 5. Visual Spot-Check
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for idx, (ctype, mdl) in enumerate(zip(
    ["salt", "blur", "occlusion"], [salt_model, blur_model, occl_model]
)):
    mdl.eval()
    corrupted, clean = PetDataset(mode="val", corruption_mode=ctype, seed=SEED)[0]
    with torch.no_grad():
        recon = mdl(corrupted.unsqueeze(0).to(device)).squeeze(0).cpu()
    for ax, image, title in zip(
        axes[idx], [corrupted, recon, clean], ["Corrupted", "Restored", "Target"]
    ):
        ax.imshow(image.permute(1, 2, 0).numpy().clip(0, 1))
        ax.set_title(f"{ctype}: {title}")
        ax.axis("off")
plt.tight_layout()
plt.show()
